# Site2Tools — control base vs LoRA en T4 (Colab)

El proposito de este cuaderno NO es entrenar nada nuevo: es **repetir en otro hardware** la tabla de control que se midio en la MI300X (exactitud de accion, modelo base vs adapter v4, sobre las tres distribuciones). Si los numeros no se sostienen en una T4, la tabla del repo dependia de la GPU y no del modelo.

Coste estimado: 12-20 min con los limites por defecto. Cada celda imprime el numero en claro.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%pip install -q 'transformers==5.17.0' 'peft==0.21.0' 'accelerate==1.15.0'
# NO se toca torch: Colab ya trae un torch con CUDA y reinstalarlo suele romper el entorno.
import torch, transformers, peft
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| transformers', transformers.__version__, '| peft', peft.__version__)

In [ ]:
%cd /content
!rm -rf ai-test && git clone -q https://github.com/dbqp01/ai-test.git && %cd /content/ai-test
!git log --oneline -1 && ls data/*.jsonl | head

## Las tres distribuciones

- `v3.valid.jsonl`: sintetica juguetes (k=2-3)
- `mind2web.valid.jsonl`: Mind2Web plano (k=7-10)
- `m2w2.valid.jsonl`: realista con 20 candidatos y split por sitio

`--adapter none` deja el modelo base delante: es el control que descarta que el gain sea del fundamental. El azar para k candidatos es 1/k, y el eval lo imprime.

In [ ]:
import subprocess, sys
ADAPTERS = {'BASE': None, 'v4': 'artifacts/site2tools-student-17b-v4'}
SPLITS = ['data/v3.valid.jsonl', 'data/mind2web.valid.jsonl', 'data/m2w2.valid.jsonl']
for split in SPLITS:
    for etiqueta, adapter in ADAPTERS.items():
        cmd = [sys.executable, 'eval_action_acc.py', '--valid', split, '--limit', '150']
        if adapter:
            cmd += ['--adapter', adapter]
        print('===', split, etiqueta, '===', flush=True)
        r = subprocess.run(cmd, capture_output=True, text=True)
        print((r.stdout or r.stderr)[-1000:], flush=True)

## Opcional: reproduccion del LoRA en T4

Si lo de arriba sostiene la tabla, esta celda sirve para comprobar que el entrenamiento es
portable (1 epoch, misma receta que `launch_v3.sh`/`v4`). No es un modelo nuevo: no hay datos
nuevos todavia (las filas `finish` por hindsight siguen sin construirse y los 6 objetivos extra
no pasan el umbral del pipeline).

In [ ]:
!python train_lora.py --model Qwen/Qwen3-1.7B --train data/v3.train.jsonl --valid data/v3.valid.jsonl --output /content/student-17b-t4 --epochs 1 --batch-size 4 --gradient-accumulation 4 --max-length 1536 2>&1 | tail -20

In [ ]:
!du -sh /content/student-17b-t4 2>/dev/null; from google.colab import drive; print('para guardar: drive.mount("/content/drive") y copiar a /content/drive/MyDrive/')